# Lab 08 — Agentic AI RCA Capstone

In [ ]:
from pathlib import Path
from IPython.display import SVG

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "environment.yml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SVG(filename=str(PROJECT_ROOT / "labs/self-study/diagrams/lab08_pipeline_position.svg"))
(PROJECT_ROOT / "outputs" / "workshop").mkdir(parents=True, exist_ok=True)

## Lab 總覽

### 學習目標
1. 理解端對端 AIOps 架構：偵測 → 預測 → 診斷 → Agentic AI
2. 整合前 Lab 的偵測管線，輸出結構化事件清單
3. 實作簡易趨勢預測 + 早期預警機制
4. 建構上下文（Incident Context）並呼叫 LLM 進行 RCA
5. 理解 Agentic AI 的工具呼叫循環，以及人類監督的必要性

### 前置條件
- 已完成 Lab 01、Lab 02（或接受 fallback 資料）
- `anthropic` SDK（選用，無 API key 時會使用 mock 回應）
- `pandas`、`matplotlib`、`numpy` 已安裝

## 設計主線：讓 Agent 使用工具，而不是取代工程師

本章把前面的採集、特徵、偵測與預測整合成 RCA context。Agentic AI 的價值在於快速查工具、整理 evidence、提出下一步，而不是直接執行高風險操作。

請用三個實務問題檢查 agent architecture：

1. **工具邊界是否清楚**：LLM 可以呼叫查詢工具，但不能憑空 invent metrics 或自行修改設備。
2. **context 是否結構化**：事件摘要、baseline、z-score、forecast breach、相關設備要先由程式整理，再交給 LLM。
3. **human gate 是否明確**：所有會影響服務的行動，例如封鎖流量、重啟服務、關閉 port，都必須由人確認。

設計原則：Agent 是 incident workflow 的協調層，不是 autonomous operator。


## 本章在 Pipeline 的位置

圖表說明：本章（Lab 08）是 Pipeline 的終點，接收 Labs 00–07 各階段的輸出：採集層的原始指標、特徵層的計算結果、偵測層的事件清單，以及其他講師介紹的進階偵測方法的輸出。

Lab 08 把這些信號整合進一個 Agentic AI 循環：偵測事件 → 上下文建構 → Anthropic claude-haiku tool_use → 可執行事件摘要。Agent 能快速提出行動建議，但執行前必須通過人類確認（人類驗證點 #8）。

---
**概念說明 ▸ 講師引導** — 請聆聽講師說明端對端 AIOps 架構。

---

## 概念：端對端 AIOps 架構

（見下方 Pipeline 架構圖）

---

### 每層的職責與資料流

```
Raw metrics (Prometheus)
    │  node_network_receive_bytes_total, ...
    ▼
[Lab 00] Collection layer
    │  PromQL query → time series DataFrame
    ▼
[Lab 01] Feature engineering
    │  rate(), rolling_mean, rolling_std, lag_12, tx_ratio, error_rate
    ▼
[Lab 02] Detection layer
    │  Z-score flag, change_point flag → event list (timestamp, device, type, severity)
    ▼
[Lab 08] Forecast layer
    │  rolling_forecast() → forecast_breach (True/False), eta_minutes
    ▼
[Lab 08] Context builder
    │  build_incident_context() → structured JSON (metrics, stats, events)
    ▼
[Lab 08] Agentic AI (LLM + Tool Calling)
    │  Tool calls: get_recent_metrics(), check_related_devices(), get_historical_events()
    ▼
[Lab 08] Output
    ├──  RCA report (natural language)
    ├──  Grafana Annotation JSON
    └──  Structured incident summary (for ticketing systems)
```

---

### 每層的職責

| 層級 | 輸入 | 輸出 | 關鍵技術 |
|------|------|------|----------|
| 採集層 | 設備 /metrics endpoint | 時序 DataFrame | PromQL, node_exporter |
| 特徵工程 | 原始 Counter 時序 | 可比較的特徵向量 | rate(), rolling stats, lag |
| 偵測層 | 特徵向量 | 事件清單（結構化）| Z-score, Change point |
| 預測層 | 最近趨勢 | 未來 15min 預警 | 線性外推（趨勢斜率）|
| 上下文建構 | 事件 + 歷史統計 | 結構化 JSON 提示詞 | 特徵聚合、規則 |
| Agentic AI | 結構化上下文 | RCA + 行動建議 | LLM + Tool Use API |

---

### 為什麼需要「上下文建構器」這一層？

直接把時序 DataFrame 丟給 LLM 不可行：
- 1 小時的資料 = 240 筆 × 10 個欄位 = 2400 個數字
- LLM 的數值理解能力不如統計計算可靠
- Token 成本高，且重要信號被大量數字稀釋

**上下文建構器的工作：**
```python
# Translate raw data into the language LLM can reason about effectively
{
  "event_time": "2024-01-15 14:32:00",
  "device": "en0",
  "current_traffic_mbps": 12.4,          # current value
  "baseline_traffic_mbps": 4.8,           # baseline (1-hour rolling mean)
  "z_score": 4.2,                         # statistical summary, not the raw series
  "duration_minutes": 8,                  # duration
  "forecast_breach_in_minutes": 5,        # forecast result
  "related_devices_also_anomalous": [],   # tool query result
  "similar_past_events": [...]            # historical events
}
```

這個 JSON 是 LLM 的「簡報」，讓它能快速聚焦在重要信號上。


In [ ]:
from pathlib import Path
from IPython.display import SVG

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "environment.yml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SVG(filename=str(PROJECT_ROOT / "labs/workshop/diagrams/lab08_end_to_end_pipeline.svg"))

---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 0：環境準備

In [ ]:
import urllib.request
import urllib.parse
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 11

PROM = "http://localhost:9090"

# 嘗試載入 Anthropic SDK
try:
    import anthropic
    HAS_ANTHROPIC = True
    print("✅ anthropic SDK 已安裝")
except ImportError:
    HAS_ANTHROPIC = False
    print("ℹ anthropic SDK 未安裝 — 使用 mock 回應")
    print("  安裝指令：pip install anthropic")

has_api_key = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"ANTHROPIC_API_KEY：{'已設定' if has_api_key else '未設定 — 使用 mock 回應'}")

def prom_range(expr, hours=6, step="1m"):
    end = datetime.utcnow()
    start = end - timedelta(hours=hours)
    params = urllib.parse.urlencode({
        "query": expr,
        "start": start.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "end":   end.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "step":  step,
    })
    with urllib.request.urlopen(f"{PROM}/api/v1/query_range?{params}", timeout=8) as r:
        return json.loads(r.read())

def to_df(result, value_col="value"):
    rows = []
    for series in result["data"]["result"]:
        dev = series["metric"].get("device",
              series["metric"].get("instance", "?"))
        for ts, val in series["values"]:
            rows.append({"timestamp": pd.Timestamp(ts, unit="s"),
                          "device": dev, value_col: float(val)})
    return pd.DataFrame(rows)

print("\n✅ 環境準備完成")

---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 1：整合偵測 Pipeline — 從 Prometheus 拉最新 2 小時資料

### 這一 Step 做什麼？

把 Lab 02 設計和驗證的偵測管線「重新組裝」成一個函數，對最新資料執行：

```
Prometheus (latest 2 hours)
    │
    ▼ prom_range() or synthetic data fallback
    │
    ▼ feature engineering
      - rx_rate, tx_rate, traffic_total
      - rolling_mean, rolling_std
    │
    ▼ detection
      - Z-score flag  (spike detection)
      - Change point flag  (regime shift)
      - Deadband debounce  (DEADBAND_N consecutive confirmations)
    │
    ▼ event list
      [{"timestamp": ..., "device": ..., "type": "z_score|change_point", ...}]
```

**與 Lab 02 的差異：**
Lab 02 是在合成資料上「設計和調參」；
這一 Step 是把相同的管線用在「最新的真實（或模擬）資料」上。
偵測參數（WINDOW_Z、Z_THRESH、DEADBAND_N 等）沿用你在 Lab 02 驗證過的值。

**執行後觀察：**
- 偵測到幾個事件？類型是 Z-score 還是 change point？
- 視覺化圖表中，偵測標記的位置是否對應你預期的異常？


In [ ]:
def generate_capstone_data(hours=2):
    """Capstone 合成資料：含明確的複合異常事件。"""
    np.random.seed(888)
    n = hours * 60
    end = datetime.utcnow()
    timestamps = [pd.Timestamp(end - timedelta(minutes=n - i)) for i in range(n)]
    t = np.arange(n)
    base = 3_000_000 + 500_000 * np.sin(t * 0.05)
    noise = np.random.normal(0, 200_000, n)
    rx = np.maximum(base + noise, 0)
    tx = rx * 0.2 + np.random.normal(0, 30_000, n)
    
    # 事件 1：高嚴重度突刺（t=40-42）
    rx[40:43] *= 9
    # 事件 2：持續high traffic（t=80-95）— 狀態切換
    rx[80:96] *= 4
    tx[80:96] *= 8  # Transmit增加更多 → 不對稱
    # 事件 3：Traffic驟降（t=100-104）
    rx[100:105] *= 0.02
    
    return pd.DataFrame({
        'timestamp': timestamps, 'device': 'eth0',
        'rx_rate': rx, 'tx_rate': np.maximum(tx, 0),
    })

filter_expr = 'device!~"lo|docker.*|veth.*|br-.*"'

# 從 Lab 02 事件檔、Prometheus、或合成資料取得偵測基礎
try:
    r_rx = prom_range(
        f'rate(node_network_receive_bytes_total{{{filter_expr}}}[1m])',
        hours=2, step='1m'
    )
    r_tx = prom_range(
        f'rate(node_network_transmit_bytes_total{{{filter_expr}}}[1m])',
        hours=2, step='1m'
    )
    df_rx = to_df(r_rx, 'rx_rate')
    df_tx = to_df(r_tx, 'tx_rate')
    df = df_rx.merge(df_tx[['timestamp', 'device', 'tx_rate']],
                     on=['timestamp', 'device'], how='left').fillna(0)
    print(f"✅ Prometheus：取得 {len(df)} 筆資料")
except Exception as e:
    print(f"⚠ Prometheus 無法連線（{e}）— 使用 Capstone 合成資料")
    df = generate_capstone_data(hours=2)
    print(f"   合成資料：{len(df)} 筆（含 3 個異常事件）")

DEVICE = df['device'].value_counts().index[0]
df = df[df['device'] == DEVICE].copy().sort_values('timestamp').reset_index(drop=True)
df['traffic_total'] = df['rx_rate'] + df['tx_rate']

print(f"分析裝置：{DEVICE}，筆數：{len(df)}")

In [ ]:
# 完整偵測管線（整合 Lab 02 方法）
WINDOW_Z    = 20
Z_THRESH    = 3.0
DEADBAND_N  = 3
SHORT_WIN   = 6
LONG_WIN    = 30
DIV_THRESH  = 0.30

# Z-score detection
df['roll_mean'] = df['traffic_total'].rolling(WINDOW_Z, min_periods=3).mean()
df['roll_std']  = df['traffic_total'].rolling(WINDOW_Z, min_periods=3).std().fillna(1)
df['z_score']   = (df['traffic_total'] - df['roll_mean']) / (df['roll_std'] + 1e-9)
raw_flag = df['z_score'].abs() > Z_THRESH
df['flag_zscore'] = (
    raw_flag.astype(int).rolling(DEADBAND_N, min_periods=DEADBAND_N).sum() >= DEADBAND_N
)

# change point detection
df['ma_short'] = df['rx_rate'].rolling(SHORT_WIN, min_periods=3).mean()
df['ma_long']  = df['rx_rate'].rolling(LONG_WIN, min_periods=10).mean()
df['ma_div']   = (df['ma_short'] - df['ma_long']).abs() / (df['ma_long'].abs() + 1e-9)
df['flag_cp']  = df['ma_div'] > DIV_THRESH

# tx_ratio 異常（TransmitRatio過高）
df['tx_ratio'] = df['tx_rate'] / (df['traffic_total'] + 1e-9)
df['flag_tx_asym'] = df['tx_ratio'] > 0.6  # Transmit超過 60%

total_flags = df['flag_zscore'] | df['flag_cp'] | df['flag_tx_asym']
print(f"整合偵測結果：")
print(f"  Z-score anomaly：{df['flag_zscore'].sum()} 筆")
print(f"  change point detection：    {df['flag_cp'].sum()} 筆")
print(f"  transmit asymmetry：  {df['flag_tx_asym'].sum()} 筆")
print(f"  任意觸發：    {total_flags.sum()} 筆")

In [ ]:
# 視覺化整合偵測
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

# Traffic圖 + 異常標記
ax = axes[0]
ax.plot(df['timestamp'], df['rx_rate'] / 1e6, color='steelblue', linewidth=1, alpha=0.7, label='RX')
ax.plot(df['timestamp'], df['tx_rate'] / 1e6, color='tomato', linewidth=1, alpha=0.7, label='TX')
ax.plot(df['timestamp'], df['roll_mean'] / 1e6, color='navy', linewidth=2, alpha=0.6, label='Rolling mean')
z_pts = df[df['flag_zscore']]
if len(z_pts): ax.scatter(z_pts['timestamp'], z_pts['rx_rate'] / 1e6, color='red', s=60, zorder=5, label='Z-score anomaly')
ax.set_ylabel('MB/s')
ax.set_title('Integrated detection pipeline')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Z-score
ax2 = axes[1]
ax2.plot(df['timestamp'], df['z_score'], color='darkorange', linewidth=1)
ax2.axhline(Z_THRESH, color='red', linestyle='--', alpha=0.6)
ax2.axhline(-Z_THRESH, color='red', linestyle='--', alpha=0.6)
ax2.fill_between(df['timestamp'], df['z_score'], 0,
                  where=df['flag_zscore'], alpha=0.3, color='red')
ax2.set_ylabel('Z-score')
ax2.set_title('Z-score')
ax2.grid(True, alpha=0.3)

# TX ratio
ax3 = axes[2]
ax3.plot(df['timestamp'], df['tx_ratio'], color='purple', linewidth=1.5)
ax3.axhline(0.6, color='red', linestyle='--', alpha=0.6, label='60% warning line')
ax3.fill_between(df['timestamp'], df['tx_ratio'], 0.6,
                  where=df['flag_tx_asym'], alpha=0.3, color='purple', label='transmit anomaly')
ax3.set_ylabel('TX Ratio')
ax3.set_title('Transmit asymmetry (TX ratio)')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

plt.tight_layout()
plt.show()

In [ ]:
# 建立事件清單（去重複、合併連續事件）
events = []

# Z-score 事件：取每段連續異常的起始點
prev = False
for _, row in df.iterrows():
    if row['flag_zscore'] and not prev:
        events.append({
            'timestamp': row['timestamp'],
            'device': DEVICE,
            'type': 'zscore',
            'severity': 'high' if abs(row['z_score']) > 5 else 'medium',
            'z_score': round(row['z_score'], 2),
            'rx_rate': row['rx_rate'],
            'tx_rate': row['tx_rate'],
            'traffic_total': row['traffic_total'],
        })
    prev = row['flag_zscore']

# change point detection events
prev = False
for _, row in df.iterrows():
    if row['flag_cp'] and not prev:
        events.append({
            'timestamp': row['timestamp'],
            'device': DEVICE,
            'type': 'changepoint',
            'severity': 'medium',
            'z_score': 0,
            'rx_rate': row['rx_rate'],
            'tx_rate': row['tx_rate'],
            'traffic_total': row['traffic_total'],
        })
    prev = row['flag_cp']

event_columns = ['timestamp', 'device', 'type', 'severity', 'z_score', 'rx_rate', 'tx_rate', 'traffic_total']
if events:
    df_events = pd.DataFrame(events, columns=event_columns).sort_values('timestamp').reset_index(drop=True)
    print(f"事件清單（{len(df_events)} 個獨立事件）：")
    print(df_events[['timestamp', 'type', 'severity', 'z_score']].to_string(index=False))
else:
    df_events = pd.DataFrame(columns=event_columns)
    print("事件清單（0 個獨立事件）：目前 live metrics 沒有觸發異常。")
    print("後續 RCA 步驟會使用示範上下文，讓 capstone 流程仍可完整執行。")


---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 2：滾動預測 + 早期預警

### 為什麼需要「預測」這一層？

偵測是回顧性的（異常已經發生才告警）；預測能在問題爆發前採取行動：

```
Without forecast (detection only):   With forecast:

  Traffic                              Traffic
  ↑              SLA breach            ↑
  │           ╱   │                    │        ╱── forecast line
  │──────────╱────┤ ← alert            │───────╱ ← early warning (15 min ahead)
  │      ───╱    │                    │    ──╱
  └──────────────────→ Time            └────────────────────→ Time

  Alert = problem has already happened  Alert = problem is about to happen (time to act)
```

---

### 線性外推算法

本 Lab 採用最簡單的預測方式：用最近 N 筆的**線性趨勢（斜率）**外推：

```
Formula:

  slope = (last_value − first_value_in_window) / window_length

  forecast(t) = last_value + slope × t   (t = minutes into the future)

Example:
  Traffic over the past 12 minutes: 4.0 → 5.2 → 6.4 MB/s
  slope = (6.4 − 4.0) / 12 = 0.2 MB/s per minute

  Forecast 15 minutes ahead: 6.4 + 0.2 × 15 = 9.4 MB/s
  If SLA upper bound = 8 MB/s → breach expected in 8 minutes
```

---

### 這個方法的限制

線性外推假設趨勢會持續，實際流量往往不是這樣：

```
Actual possible outcomes:
  ╱── actually continues to rise    ← linear extrapolation correct
  ╱─── drops after the peak          ← over-warning
  ╱─── suddenly levels off            ← over-warning

Applicable when:
  - Event is still in progress (Z-score remains high)
  - Trend within the observation window is stable (no violent fluctuation)
  - Forecast horizon does not exceed 30 minutes (longer → less reliable)
```

更複雜的方法（課後自學）：ARIMA、Prophet、LSTM——但對於 AIOps 的早期預警場景，
線性外推的準確度往往已經足夠，且不需要訓練期。


In [ ]:
def rolling_forecast(series, horizon=6):
    """
    簡易趨勢外推預測。
    series: 最近的時序資料（Series）
    horizon: 預測步數
    回傳：未來 horizon 步的預測值清單
    """
    if len(series) < 2:
        return [series.iloc[-1]] * horizon
    recent = series.iloc[-12:] if len(series) >= 12 else series
    slope = (float(recent.iloc[-1]) - float(recent.iloc[0])) / max(len(recent) - 1, 1)
    last_val = float(recent.iloc[-1])
    return [max(last_val + slope * (i + 1), 0) for i in range(horizon)]

# 為每個事件計算預測
if len(df_events) == 0:
    print("沒有偵測到事件，跳過預測步驟")
else:
    FORECAST_HORIZON = 6  # 預測 6 分鐘後
    ALERT_THRESHOLD = df['traffic_total'].rolling(WINDOW_Z).mean().dropna().quantile(0.90)
    
    print(f"預警threshold（P90 rolling mean）：{ALERT_THRESHOLD / 1e6:.2f} MB/s")
    print()
    
    forecast_results = []
    for idx, event in df_events.iterrows():
        # 取事件前的時序資料
        ts_event = event['timestamp']
        hist = df[df['timestamp'] <= ts_event]['traffic_total'].dropna()
        if len(hist) < 3:
            continue
        
        forecast = rolling_forecast(hist, horizon=FORECAST_HORIZON)
        forecast_breach = any(v > ALERT_THRESHOLD for v in forecast)
        
        forecast_results.append({
            'event_idx': idx,
            'timestamp': ts_event,
            'type': event['type'],
            'severity': event['severity'],
            'rx_rate_now': event['rx_rate'],
            'forecast_6min_max': max(forecast) / 1e6,
            'forecast_breach': forecast_breach,
        })
    
    df_forecast = pd.DataFrame(forecast_results)
    if len(df_forecast) > 0:
        print("事件預測結果：")
        print(df_forecast[['timestamp', 'type', 'severity', 'forecast_6min_max', 'forecast_breach']]
              .to_string(index=False))

In [ ]:
# 視覺化第一個事件的預測
if len(df_events) > 0:
    event_ts = df_events['timestamp'].iloc[0]
    hist_df = df[df['timestamp'] <= event_ts].tail(30)
    forecast_vals = rolling_forecast(hist_df['traffic_total'], horizon=FORECAST_HORIZON)
    
    future_ts = [event_ts + timedelta(minutes=i+1) for i in range(FORECAST_HORIZON)]
    
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(hist_df['timestamp'], hist_df['traffic_total'] / 1e6,
            color='steelblue', linewidth=2, label='historical traffic')
    ax.plot(future_ts, [v / 1e6 for v in forecast_vals],
            color='red', linewidth=2, linestyle='--', marker='o', markersize=5, label='forecast')
    ax.axhline(ALERT_THRESHOLD / 1e6, color='red', linestyle=':', alpha=0.6, label='early-warning threshold')
    ax.axvline(event_ts, color='orange', linewidth=2, label='event timestamp')
    ax.fill_between([event_ts] + future_ts,
                     0, [df['traffic_total'].max()/1e6] * (FORECAST_HORIZON + 1),
                     alpha=0.1, color='red', label='forecast period')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.set_ylabel('Traffic (MB/s)')
    ax.set_title(f'Rolling forecast: after the first event {FORECAST_HORIZON} minute forecast')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    breach = any(v > ALERT_THRESHOLD for v in forecast_vals)
    print(f"預測結論：{'⚠ 預計 {FORECAST_HORIZON} 分鐘內將突破threshold！' if breach else '✅ 預測Traffic在threshold以下'}")
else:
    print("沒有事件，跳過預測視覺化")

---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 3：上下文建構器（Incident Context Builder）

### LLM 的效果高度依賴「輸入品質」

同一個異常事件，兩種輸入方式給 LLM 的效果天差地別：

**差的輸入（原始數據）：**
```
Here is the network traffic data for the past 2 hours:
time,traffic
14:00,4.2
14:01,4.3
... (240 rows)
Please analyze the root cause.
```
問題：LLM 需要自己做統計計算，容易出錯；重要信號被大量數字稀釋。

**好的輸入（結構化上下文）：**
```json
{
  "event": {
    "device": "en0",
    "detected_at": "2024-01-15 14:32",
    "duration_minutes": 8
  },
  "current_state": {
    "traffic_mbps": 12.4,
    "z_score": 4.2,
    "error_rate_pct": 0.8
  },
  "baseline": {
    "mean_mbps": 4.8,
    "std_mbps": 1.8,
    "p95_mbps": 7.2
  },
  "forecast": {
    "breach_predicted": true,
    "eta_minutes": 8
  },
  "correlated": {
    "other_devices_anomalous": [],
    "recent_config_changes": []
  }
}
```
LLM 只需要「理解和推理」，統計計算已由 Python 完成。

---

### 上下文建構器的三個設計原則

**1. 摘要，不是原始資料**
把 240 筆數字壓縮成 mean、std、p95、當前值、Z-score。

**2. 包含對比（偏差量化）**
「目前 12.4 MB/s」 vs「基線 4.8 MB/s」比單純「12.4 MB/s」有意義得多。

**3. 填入所有可用的相關性信號**
相關設備、歷史事件、預測結果——讓 LLM 能做跨維度推理，而不只是看單一指標。

---

### 實作說明

下方 `build_incident_context()` 函數接收事件和 DataFrame，輸出標準化的上下文 JSON。
這個 JSON 會直接被 Step 4 的 Agentic AI 使用。

執行時觀察輸出的 JSON 結構：每個欄位都是 LLM 診斷所需資訊的一部分。
試著思考：如果拿掉其中某個欄位，LLM 的診斷品質會如何下降？


In [ ]:
def compute_baseline_stats(df, window_before=60):
    """計算事件前的基線統計。"""
    baseline = df.tail(window_before)
    return {
        'rx_rate_mean_mb': round(baseline['rx_rate'].mean() / 1e6, 3),
        'rx_rate_std_mb':  round(baseline['rx_rate'].std()  / 1e6, 3),
        'tx_ratio_mean':   round(baseline['tx_rate'].sum() / (baseline['traffic_total'].sum() + 1e-9), 3),
    }

def compute_duration_minutes(df, event_ts, flag_col):
    """估算異常duration（分鐘）。"""
    after = df[df['timestamp'] >= event_ts]
    if flag_col not in after.columns:
        return 0
    sustained = after[flag_col].astype(int).cumsum()
    end_idx = sustained[sustained == sustained.max()].index
    if len(end_idx) == 0:
        return 0
    return int((after.loc[end_idx[-1], 'timestamp'] - event_ts).total_seconds() / 60)

def build_incident_context(event, df):
    """建構完整的事件上下文供 LLM 診斷。"""
    ts = event['timestamp']
    baseline_stats = compute_baseline_stats(df[df['timestamp'] < ts])
    
    flag_map = {'zscore': 'flag_zscore', 'changepoint': 'flag_cp'}
    flag_col = flag_map.get(event['type'], 'flag_zscore')
    duration = compute_duration_minutes(df, ts, flag_col)
    
    # 同時發生的其他異常信號
    concurrent = []
    row = df[df['timestamp'] == ts]
    if not row.empty:
        r = row.iloc[0]
        if r.get('flag_tx_asym', False): concurrent.append('transmit asymmetry（TX ratio > 60%）')
        if r.get('flag_cp', False) and event['type'] != 'changepoint': concurrent.append('同時偵測到change point')
        if r.get('flag_zscore', False) and event['type'] != 'zscore': concurrent.append('同時偵測到 Z-score anomaly')
    
    forecast_breach = False
    if 'df_forecast' in globals():
        fc_row = df_forecast[df_forecast['timestamp'] == ts]
        if not fc_row.empty:
            forecast_breach = bool(fc_row.iloc[0].get('forecast_breach', False))
    
    return {
        'device': event['device'],
        'event_time': str(ts),
        'anomaly_type': event['type'],
        'severity': event['severity'],
        'rx_rate_now_mb': round(event['rx_rate'] / 1e6, 3),
        'rx_rate_baseline_mb': baseline_stats['rx_rate_mean_mb'],
        'rx_rate_baseline_std_mb': baseline_stats['rx_rate_std_mb'],
        'rx_deviation_x': round(
            event['rx_rate'] / (baseline_stats['rx_rate_mean_mb'] * 1e6 + 1e-9), 1
        ),
        'z_score': event.get('z_score', 0),
        'tx_ratio_now': round(event['tx_rate'] / (event['traffic_total'] + 1e-9), 3),
        'tx_ratio_baseline': baseline_stats['tx_ratio_mean'],
        'duration_minutes': duration,
        'forecast_breach_6min': forecast_breach,
        'concurrent_signals': concurrent,
        'analysis_timestamp': datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
    }

# 為最嚴重的事件建構上下文
if len(df_events) > 0:
    sev_order = {'high': 0, 'medium': 1, 'low': 2}
    top_event = df_events.sort_values('severity', key=lambda s: s.map(sev_order)).iloc[0].to_dict()
    incident_ctx = build_incident_context(top_event, df)
    
    print("最高嚴重度事件的上下文：")
    print(json.dumps(incident_ctx, ensure_ascii=False, indent=2))
else:
    # fallback 上下文
    incident_ctx = {
        'device': DEVICE,
        'event_time': str(datetime.utcnow()),
        'anomaly_type': 'zscore',
        'severity': 'high',
        'rx_rate_now_mb': 25.5,
        'rx_rate_baseline_mb': 3.0,
        'rx_deviation_x': 8.5,
        'z_score': 7.2,
        'duration_minutes': 3,
        'forecast_breach_6min': True,
        'concurrent_signals': ['transmit asymmetry'],
        'analysis_timestamp': datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
    }
    print("使用示範上下文：")
    print(json.dumps(incident_ctx, ensure_ascii=False, indent=2))

---
**概念說明 ▸ 講師引導** — 請聆聽講師說明 Agentic AI vs 簡單 LLM 的差別。

---

## 概念：Agentic AI vs 簡單 LLM 呼叫

（見下方比較圖）

### 在 AIOps 中，工具（Tools）是什麼？

```
Tool name                     What it does

get_recent_metrics()    →   query latest Prometheus metrics
check_related_devices() →   check whether other devices are also anomalous (correlation)
get_historical_events() →   query historical alert records
create_incident_ticket()→   create a ticket in JIRA/ServiceNow
send_alert()            →   send PagerDuty / Slack notification
```

**關鍵原則：AI 可以「建議」執行哪個工具，但「執行」必須有人確認。**

**參考資料：**
- Anthropic Tool Use API：https://docs.anthropic.com/en/docs/build-with-claude/tool-use
- Anthropic Claude API 定價：https://www.anthropic.com/pricing

In [ ]:
from pathlib import Path
from IPython.display import SVG

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "environment.yml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SVG(filename=str(PROJECT_ROOT / "labs/workshop/diagrams/lab08_agentic_vs_llm.svg"))

---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 4：Agentic AI RCA 循環

### 「Agentic」是什麼意思？

一般 LLM 呼叫（一問一答）：

```
You → [question] → LLM → [answer]
              (single-turn; LLM reasons only from the input text)
```

Agentic AI（多輪 + 工具呼叫）：

```
You → [question + tool definitions] → LLM
                                          │
                                          ├─ decides to call get_recent_metrics()
                                          │    └→ execute tool → retrieve real-time data
                                          │    └→ return result to LLM
                                          │
                                          ├─ decides to call check_related_devices()
                                          │    └→ execute tool → confirm whether anomaly is spreading
                                          │    └→ return result to LLM
                                          │
                                          └─ integrate all tool results → generate final RCA report
```

Agent 的「主動性」在於它能**自己決定要問什麼問題**（呼叫哪個工具），而不是等你告訴它。

---

### Tool Calling 的技術機制

```python
# Tool definitions (tell the LLM which tools are available)
TOOLS = [
  {
    "name": "get_recent_metrics",
    "description": "Query traffic metrics for the past N minutes",
    "input_schema": {
      "device": "str",
      "minutes": "int"
    }
  },
  ...
]

# The LLM response may contain a tool_use block
response = {
  "type": "tool_use",
  "name": "get_recent_metrics",
  "input": {"device": "en0", "minutes": 30}
}

# Your code executes the tool and returns the result to the LLM
result = execute_tool(response)
# After seeing the result, the LLM may call another tool or generate the final response
```

---

### 本 Lab 的 RCA 循環步驟

1. **偵測事件** — Step 1 的 event list 提供觸發信號
2. **建構上下文** — Step 3 的 `build_incident_context()` 把統計摘要打包成 JSON
3. **呼叫 Agent** — `run_rca_agent()` 送出提示詞 + 工具定義
4. **工具執行** — Agent 呼叫工具，Python 側執行並回傳結果
5. **迭代** — Agent 根據工具結果決定是否再呼叫其他工具
6. **輸出** — 最終生成 RCA 報告 + 行動建議

---

### 如果沒有 Anthropic API Key

本 Lab 預設使用 Mock 模式：用預先準備好的回應模擬 Agent 行為。
Mock 模式讓你能看到完整的輸出格式和工具呼叫結構，即使沒有 API Key 也能完整執行。

要使用真實 API，在環境變數中設定 `ANTHROPIC_API_KEY` 即可自動切換。


In [ ]:
# 定義 AIOps Agent 的工具集
TOOLS = [
    {
        "name": "get_recent_metrics",
        "description": "查詢 Prometheus 取得指定裝置的最近指標值。回傳最近 10 分鐘的 rx_rate、tx_rate 統計。",
        "input_schema": {
            "type": "object",
            "properties": {
                "device":  {"type": "string", "description": "網路介面名稱，例如 eth0"},
                "minutes": {"type": "integer", "description": "查詢最近幾分鐘的資料", "default": 10}
            },
            "required": ["device"]
        }
    },
    {
        "name": "check_related_devices",
        "description": "檢查同主機上其他網路介面是否也出現類似異常。用於判斷是單一介面問題還是全機問題。",
        "input_schema": {
            "type": "object",
            "properties": {
                "threshold_mb": {"type": "number", "description": "Trafficthreshold (MB/s)"}
            }
        }
    },
    {
        "name": "create_incident_summary",
        "description": "生成結構化事件報告，包含：嚴重度、根因假設（ranked list）、建議行動、是否需要升級。",
        "input_schema": {
            "type": "object",
            "properties": {
                "severity":    {"type": "string", "enum": ["low", "medium", "high", "critical"]},
                "root_causes": {"type": "array",  "items": {"type": "string"}},
                "actions":     {"type": "array",  "items": {"type": "string"}},
                "escalate":    {"type": "boolean"}
            },
            "required": ["severity", "root_causes", "actions"]
        }
    }
]

AGENT_SYSTEM_PROMPT = """你是資深的 AIOps 網路診斷工程師。
你負責分析網路監控告警，提供根因分析（RCA）和處置建議。

分析步驟：
1. 仔細閱讀提供的事件上下文
2. 若需要更多資料，使用可用工具查詢
3. 提出 2-3 個根因假設（從最可能到最不可能排列）
4. 提出具體可執行的處置步驟
5. 判斷是否需要升級（叫醒 on-call 工程師）

重要：
- 你的建議僅供參考，所有執行動作需人類確認
- 避免誤警報造成告警疲勞
- 在資訊不足時，明確說明需要哪些額外資訊"""

print("✅ Agent 工具集與 System Prompt 設定完成")
print(f"工具數量：{len(TOOLS)}")
for t in TOOLS:
    print(f"  - {t['name']}: {t['description'][:40]}...")

In [ ]:
# Mock 工具回應（當 Prometheus 不可用時）
def mock_tool_response(tool_name, tool_input):
    """模擬工具回傳值（當 Prometheus/真實環境不可用時使用）。"""
    if tool_name == "get_recent_metrics":
        device = tool_input.get("device", DEVICE)
        return json.dumps({
            "device": device,
            "window_minutes": tool_input.get("minutes", 10),
            "rx_rate_avg_mb": 24.3,
            "rx_rate_max_mb": 28.7,
            "tx_rate_avg_mb": 5.2,
            "current_time": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
            "note": "Mock data — Prometheus not available"
        })
    elif tool_name == "check_related_devices":
        return json.dumps({
            "other_devices": [
                {"device": "lo", "rx_rate_avg_mb": 0.001, "anomaly": False},
            ],
            "conclusion": f"只有 {DEVICE} 受影響，其他介面正常",
            "note": "Mock data"
        })
    elif tool_name == "create_incident_summary":
        return json.dumps({
            "recorded": True,
            "incident_id": f"INC-{datetime.utcnow().strftime('%Y%m%d%H%M')}",
            "message": "摘要已記錄"
        })
    return json.dumps({"error": f"Unknown tool: {tool_name}"})

def call_real_tool(tool_name, tool_input):
    """嘗試呼叫真實工具（此 Lab 版本皆用 mock，可替換為真實實作）。"""
    # 在生產環境中，可替換為真實的 Prometheus 查詢或票務系統 API
    return mock_tool_response(tool_name, tool_input)

print("✅ 工具處理函數設定完成（使用 mock 實作）")

In [ ]:
# Mock Agent 回應（當沒有 API key 時）
MOCK_AGENT_RESPONSE = """根因分析報告

事件概要：
裝置 {device} 在 {event_time} 發生網路Traffic異常，
receive rate達到基線的 {rx_deviation_x} 倍（Z-score = {z_score}）。

根因假設（按可能性排序）：
1. [高可能性] DDoS 或大量外部請求
   - 依據：Traffic突增達基線 8.5 倍，Z-score = 7.2
   - 驗證：檢查來源 IP 分布，確認是否有單一來源佔比過高

2. [中可能性] 備份或批次任務意外在業務Time執行
   - 依據：Traffic類型未知，可能是大量資料傳輸
   - 驗證：確認排程任務是否提前或延遲執行

3. [低可能性] 網路設備故障導致封包重傳
   - 依據：需確認錯誤率是否同步上升

建議行動：
1. 立即：執行 netstat -an | grep ESTABLISHED | wc -l 確認連線數
2. 立即：檢查防火牆日誌，識別Traffic來源
3. 5分鐘內：若確認為外部攻擊，啟動 rate limiting 規則
4. 15分鐘內：通知安全團隊進行調查

升級建議：需要（預測 6 分鐘內仍持續超標）
建議通知：on-call 網路工程師"""

def run_rca_agent(incident_context):
    """
    執行 Agentic AI RCA 循環。
    若有 API key 且已安裝 anthropic SDK，使用真實 LLM；否則使用 mock 回應。
    """
    if not HAS_ANTHROPIC or not has_api_key:
        mode = "mock SDK" if not HAS_ANTHROPIC else "mock（無 API key）"
        print(f"[Agent] 使用 {mode} 模式")
        response = MOCK_AGENT_RESPONSE.format(
            device=incident_context.get('device', 'eth0'),
            event_time=incident_context.get('event_time', '未知'),
            rx_deviation_x=incident_context.get('rx_deviation_x', 'N/A'),
            z_score=incident_context.get('z_score', 'N/A'),
        )
        return response, []

    client = anthropic.Anthropic()
    messages = [{
        "role": "user",
        "content": f"請分析以下網路事件並提供根因分析：\n\n{json.dumps(incident_context, ensure_ascii=False, indent=2)}"
    }]
    
    tool_calls_made = []
    max_iterations = 5
    iteration = 0
    
    print(f"[Agent] 開始分析... (最多 {max_iterations} 輪工具呼叫)")
    
    while iteration < max_iterations:
        iteration += 1
        print(f"[Agent] 第 {iteration} 輪")
        
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=2048,
            system=AGENT_SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages,
        )
        
        if response.stop_reason == "end_turn":
            print(f"[Agent] 分析完成（{iteration} 輪）")
            final_text = next(
                (b.text for b in response.content if hasattr(b, 'text')),
                "（無文字回應）"
            )
            return final_text, tool_calls_made
        
        if response.stop_reason == "tool_use":
            # 處理工具呼叫
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"[Agent] 呼叫工具：{block.name}({json.dumps(block.input)[:60]}...)")
                    result_content = call_real_tool(block.name, block.input)
                    tool_calls_made.append({'tool': block.name, 'input': block.input})
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result_content
                    })
            
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user",      "content": tool_results})
        else:
            print(f"[Agent] 非預期停止原因：{response.stop_reason}")
            break
    
    return "達到最大迭代次數", tool_calls_made

print("✅ RCA Agent 函數設定完成")

In [ ]:
# 執行 RCA Agent
print("執行 Agentic AI 根因分析...")
print("=" * 60)

agent_response, tool_calls = run_rca_agent(incident_ctx)

print()
print("Agent RCA 結果：")
print("-" * 60)
print(agent_response)
print("-" * 60)

if tool_calls:
    print(f"\nAgent 使用了 {len(tool_calls)} 次工具呼叫：")
    for tc in tool_calls:
        print(f"  - {tc['tool']}")

---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 5：輸出可執行事件摘要

### 把 RCA 結果轉換成可操作的輸出

Agentic AI 的回應是自然語言——維運流程需要的是結構化資料：

```
Agent natural-language output:
  "Analysis indicates a traffic surge on interface en0 at 14:32,
   Z-score reached 4.2, persisting for 8 minutes.
   Recommendations: (1) confirm whether a scheduled backup was running;
   (2) if unplanned, escalate to Network Team for verification."

   ↓ parse_incident_summary() ↓

Structured output:
{
  "incident_id": "INC-20240115-143200",
  "severity": "medium",           ← routable by PagerDuty
  "device": "en0",
  "root_cause_hypothesis": "...", ← usable for JIRA ticket
  "recommended_actions": [...],   ← usable for runbook automation
  "grafana_annotation": {...}     ← POST directly to Grafana API
}
```

### 輸出的三種去向

**即時路徑（Grafana Annotation）：**
直接 POST 到 Grafana API，在 Dashboard 上標注事件時間點。

**工單路徑（JIRA / ServiceNow）：**
解析 `recommended_actions`，自動建立工單並分派給相關 Team。

**通知路徑（Slack / PagerDuty）：**
依 severity 路由，medium → Slack，high → PagerDuty 呼叫 on-call。

本 Lab 輸出 JSON，讓你看到完整的結構；實際 POST 到外部系統需要對應的 API Token。


In [ ]:
def parse_incident_summary(agent_response_text, incident_ctx):
    """
    將 Agent 的自然語言回應轉換為結構化摘要。
    在生產環境中，通常會要求 Agent 回傳 JSON 格式。
    """
    # 簡化版：直接從上下文推斷
    sev = incident_ctx.get('severity', 'medium')
    z   = incident_ctx.get('z_score', 0)
    
    if sev == 'high' or abs(z) > 5:
        final_sev = 'HIGH'
        escalate  = True
    else:
        final_sev = 'MEDIUM'
        escalate  = incident_ctx.get('forecast_breach_6min', False)
    
    return {
        'incident_id':    f"INC-{datetime.utcnow().strftime('%Y%m%d-%H%M%S')}",
        'device':         incident_ctx.get('device', DEVICE),
        'detected_at':    incident_ctx.get('event_time'),
        'severity':       final_sev,
        'anomaly_type':   incident_ctx.get('anomaly_type'),
        'z_score':        incident_ctx.get('z_score'),
        'rx_deviation_x': incident_ctx.get('rx_deviation_x'),
        'duration_min':   incident_ctx.get('duration_minutes', 0),
        'forecast_breach':incident_ctx.get('forecast_breach_6min', False),
        'concurrent_signals': incident_ctx.get('concurrent_signals', []),
        'rca_summary':    agent_response_text[:500] + '...' if len(agent_response_text) > 500 else agent_response_text,
        'escalate':       escalate,
        'tool_calls_made': len(tool_calls),
        'pipeline_ts':    datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
    }

summary = parse_incident_summary(agent_response, incident_ctx)

# 儲存完整摘要
output_path = PROJECT_ROOT / "outputs" / "workshop" / "workshop_lab08_incident_summary.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("事件摘要（可執行版本）：")
print("=" * 60)
print(f"  事件 ID：   {summary['incident_id']}")
print(f"  裝置：      {summary['device']}")
print(f"  嚴重度：    {summary['severity']}")
print(f"  異常類型：  {summary['anomaly_type']}")
print(f"  Z-score：   {summary['z_score']}")
print(f"  偏離倍數：  {summary['rx_deviation_x']}x 基線")
print(f"  duration：  {summary['duration_min']} 分鐘")
print(f"  預測突破：  {'是' if summary['forecast_breach'] else '否'}")
print(f"  需升級：    {'⚠ 是' if summary['escalate'] else '否'}")
print(f"  工具呼叫數：{summary['tool_calls_made']}")
print()
print(f"完整摘要已儲存：{output_path}")

In [ ]:
# Grafana Annotation：將 RCA 結果推送為 Grafana 標注
grafana_annotation = {
    "time":     int(datetime.utcnow().timestamp() * 1000),
    "text":     f"[{summary['severity']}] {summary['incident_id']} — Z={summary['z_score']}",
    "tags":     ["aiops", "rca", summary['anomaly_type'], summary['device']],
    "isRegion": False,
}

print("Grafana Annotation 物件：")
print(json.dumps(grafana_annotation, ensure_ascii=False, indent=2))

# 嘗試推送到 Grafana API（若可用）
GRAFANA_URL = os.environ.get("GRAFANA_CLOUD_URL", "https://<YOUR_ORG>.grafana.net")
GRAFANA_TOKEN = os.environ.get("GRAFANA_CLOUD_TOKEN", "")  # Grafana Cloud API token
try:
    payload = json.dumps(grafana_annotation).encode('utf-8')
    req = urllib.request.Request(
        f"{GRAFANA_URL}/api/annotations",
        data=payload,
        headers={
            'Content-Type': 'application/json',
            # 在實際環境中：'Authorization': 'Bearer <grafana_api_key>'
        },
        method='POST'
    )
    with urllib.request.urlopen(req, timeout=3) as r:
        resp = json.loads(r.read())
        print(f"\n✅ Grafana Annotation 推送成功：ID = {resp.get('id')}")
except Exception as e:
    print(f"\nℹ Grafana 推送跳過（{e}）")
    print("  提示：可在 Grafana UI 手動匯入 annotation，或設定 API token 後自動推送")

---
**探索練習** — 修改 System Prompt 觀察 Agent 回應風格的變化。

---

## 探索練習：自訂 Agent System Prompt

### 為什麼 System Prompt 很重要？

System Prompt 定義了 Agent 的「個性」和「決策準則」。
同樣的事件上下文，不同的 System Prompt 可能產生截然不同的 RCA 結果。

### 實驗 1：改變 Agent 的保守 / 積極程度

```python
# Conservative (default)
CUSTOM_SYSTEM_PROMPT = """You are a cautious AIOps engineer. You assume most alerts are false positives.
Before recommending escalation, you require at least 3 independent signals to appear simultaneously."""

# Aggressive
CUSTOM_SYSTEM_PROMPT = """You are a fast-responding SRE. You believe early alerting beats missed detections.
Any event with Z-score > 3 should be escalated immediately; waiting for more confirmation is worse than acting fast."""

# Neutral (let the LLM decide)
CUSTOM_SYSTEM_PROMPT = """You are an AIOps analyst. Make the most objective judgment based on the data,
state your confidence level, and provide two options — "Recommended action" and "Watch and wait" — for the human to choose from."""
```

### 實驗 2：改變升級準則

```python
# Strict escalation criteria
CUSTOM_SYSTEM_PROMPT = """You are a strict AIOps engineer.
Recommend escalation only when ALL of the following conditions are simultaneously true:
1. Z-score > 5 (extremely large deviation)
2. The event has persisted for more than 10 minutes
3. At least 2 related devices show anomalies at the same time
Otherwise, log the event without escalating."""
```

### 觀察重點

執行不同 Prompt 後，比較：
- 根因假設的排序是否改變？
- 建議行動的強度（「監控」vs「立即隔離」）是否改變？
- LLM 有沒有遵守你的規則（如「只在 Z>5 才升級」）？

**關鍵洞察：** System Prompt 是「軟性護欄」，LLM 可能不完全遵守。
這是為什麼「人類驗證點」仍然是 Pipeline 設計的關鍵環節。


In [ ]:
# 🔧 修改這裡！
CUSTOM_SYSTEM_PROMPT = """你是謹慎的 AIOps 工程師。你認為大多數告警是假警報。
在建議升級前，你要求至少 3 個獨立信號同時出現。
你的回應應包含：信心等級（高/中/低）、根因假設（最多 2 個）、行動建議。
只使用繁體中文回應。"""

# 用自訂 prompt 再執行一次
if HAS_ANTHROPIC and has_api_key:
    client = anthropic.Anthropic()
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        system=CUSTOM_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": f"分析：{json.dumps(incident_ctx, ensure_ascii=False)}"}]
    )
    print("自訂 Prompt 回應：")
    print(resp.content[-1].text if resp.content else "無回應")
else:
    print("Mock 模式：自訂 Prompt 效果需要真實 API Key 才能觀察")
    print("但在真實環境中，以下差異會很明顯：")
    differences = [
        "謹慎 prompt → 更少的升級建議",
        "積極 prompt → 更多的預防性行動",
        "結構化 prompt → 更一致的回應格式",
    ]
    for d in differences:
        print(f"  • {d}")

---
**討論暫停 ▸ 全班討論** — 暫停執行，與全班分享你的觀察。

---

## ⚠ 人類驗證點 #8 — Agentic AI 可以快速行動，但人類必須在執行前確認

### 判斷標準

| 問題 | 你需要思考的面向 |
|------|----------------|
| Agent 的根因假設合理嗎？ | 你的業務知識是否支持這個假設？ |
| 建議的行動有副作用嗎？ | 執行 rate limiting 會影響正常用戶嗎？ |
| 升級是否必要？ | 避免告警疲勞 vs 快速反應的取捨 |
| Agent 遺漏了什麼？ | LLM 不知道你的業務背景、計畫性事件 |

---

### 自動化行動風險矩陣

不是所有 Agent 行動都一樣危險。根據**可逆性**和**影響範圍**分類：

```
                    Impact scope
               Individual   Org-wide   Customer-facing
Rev-  Low  ┌──────────────────────────────────────┐
ers-       │  ✅ Auto    │  ✅ Auto    │ ⚠️ Manual confirm │
ibi-       ├──────────────────────────────────────┤
lity  High │ ⚠️ Manual   │ ❌ Auth req │ ❌ Auth req       │
          └──────────────────────────────────────┘

Examples (low risk — can automate):
  - Create event annotation in Grafana
  - Send notification to Slack channel
  - Update monitoring Dashboard status

Examples (high risk — require human confirmation):
  - Rate-limit traffic for an IP or device
  - Restart a service or device
  - Modify firewall rules
  - Create a ticket and auto-assign to on-call
```

---

### 核心原則

```
AI's speed advantages:                 Human judgment advantages:

24/7 continuous monitoring              Business context knowledge
Millisecond-level pattern recognition   Ethical and risk judgment
Simultaneous multi-metric analysis      Authorization and accountability
Immune to alert fatigue                 Creative problem solving
                                        Planned-event awareness (backup windows, deployments)

→ Best practice: AI detects + AI diagnoses + human decides + AI executes (low-risk actions)
```

---

### 討論問題

> 「Agent 建議了什麼行動？你會直接執行嗎？為什麼？」

> 「如果你的 Agent 一天觸發 100 次，你的團隊能處理多少？如何設計才能讓 AI 幫你篩選？」

> 「哪些行動可以讓 AI 自動執行？哪些絕對需要人類批准？你的邊界在哪裡？」

> 「如果 Agent 的根因假設是錯的，你怎麼知道？你有驗證機制嗎？」

---

### 設計建議：「人類確認閘道（Human-in-the-Loop Gate）」

```python
# Recommended implementation pattern
def execute_agent_action(action, risk_level):
    if risk_level == "low":
        return auto_execute(action)   # execute automatically
    elif risk_level == "medium":
        notify_oncall(action)         # notify but do not execute
        return "pending_approval"
    elif risk_level == "high":
        require_human_approval(action)  # require human confirmation
        return "blocked_until_approved"
```


In [ ]:
# Capstone 最終視覺化：整合 Pipeline 總覽
fig = plt.figure(figsize=(14, 10))

# 上方：完整時序 + 所有偵測標記
ax1 = fig.add_subplot(3, 1, 1)
ax1.plot(df['timestamp'], df['rx_rate'] / 1e6, color='steelblue', linewidth=1, alpha=0.7, label='RX rate')
ax1.plot(df['timestamp'], df['roll_mean'] / 1e6, color='navy', linewidth=2, label='rolling baseline')

# 標記各類事件
z_events = df_events[df_events['type'] == 'zscore'] if len(df_events) > 0 else pd.DataFrame()
cp_events = df_events[df_events['type'] == 'changepoint'] if len(df_events) > 0 else pd.DataFrame()

if len(z_events) > 0:
    ax1.scatter(z_events['timestamp'], z_events['rx_rate'] / 1e6,
                color='red', s=100, zorder=5, marker='v', label=f'Z-score anomaly ({len(z_events)})')
if len(cp_events) > 0:
    for ts in cp_events['timestamp']:
        ax1.axvline(ts, color='orange', alpha=0.5, linewidth=2)
    from matplotlib.lines import Line2D
    ax1.legend(handles=ax1.get_legend_handles_labels()[0] +
               [Line2D([0], [0], color='orange', linewidth=2, label=f'change point ({len(cp_events)})')],
               labels=ax1.get_legend_handles_labels()[1] + [f'change point ({len(cp_events)})'],
               fontsize=9)
else:
    ax1.legend(fontsize=9)

ax1.set_ylabel('MB/s')
ax1.set_title('AIOps Capstone: integrated detection results')
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

# 中間：Z-score time series
ax2 = fig.add_subplot(3, 1, 2)
ax2.plot(df['timestamp'], df['z_score'], color='darkorange', linewidth=1)
ax2.axhline(Z_THRESH, color='red', linestyle='--', alpha=0.5, label=f'Z=±{Z_THRESH}')
ax2.axhline(-Z_THRESH, color='red', linestyle='--', alpha=0.5)
ax2.fill_between(df['timestamp'], df['z_score'], 0, where=df['z_score'].abs() > Z_THRESH,
                  alpha=0.3, color='red')
ax2.set_ylabel('Z-score')
ax2.set_title('Z-score detection layer')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

# 下方：TX ratio 不對稱
ax3 = fig.add_subplot(3, 1, 3)
ax3.plot(df['timestamp'], df['tx_ratio'], color='purple', linewidth=1.5, label='TX Ratio')
ax3.axhline(0.6, color='red', linestyle='--', alpha=0.5, label='60% warning line')
ax3.fill_between(df['timestamp'], df['tx_ratio'], 0.6,
                  where=df['flag_tx_asym'], alpha=0.3, color='purple')
ax3.set_ylabel('TX Ratio')
ax3.set_title('Transmit asymmetry detection layer')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

plt.tight_layout()
_img_path = PROJECT_ROOT / "outputs" / "workshop" / "workshop_lab08_capstone_dashboard.png"
plt.savefig(str(_img_path), dpi=150, bbox_inches='tight')
plt.show()
print(f"\n圖表已儲存到 {_img_path}")

In [ ]:
# 最終工作坊總結
print("="*60)
print("AIOps 實戰工作坊 — 最終總結")
print("="*60)
print()
print("完成的 Lab：")
print("  Lab 00：Prometheus 架構理解 + PromQL 基礎")
print("  Lab 01：時序特徵工程（Counter→Rate, lag, rolling, multi-res）")
print("  Lab 02：三種異常偵測方法（threshold、Z-score、change point）")
print("  Lab 08：端對端 AIOps pipeline + Agentic AI RCA")
print()
print("你已建立的能力：")
capabilities = [
    "從 Prometheus 拉取真實指標並做基本視覺化",
    "設計反映業務意義的複合特徵",
    "實作並比較三種異常偵測方法",
    "理解 Agentic AI 工具呼叫循環",
    "建構結構化事件上下文供 LLM 診斷",
    "輸出可供 Grafana 使用的事件標注",
]
for cap in capabilities:
    print(f"  ✅ {cap}")
print()
print("關鍵學習：")
print("  1. 偵測方法的選擇取決於業務場景，沒有萬能方案")
print("  2. threshold是業務決策，AI 只能提供統計參考")
print("  3. Agentic AI 加速診斷，但人類必須做最終決策")
print("  4. 告警品質 > 告警數量 — 寧少而精")
print()
print("下一步自學資源：")
resources = [
    ("Prometheus 官方文件",   "https://prometheus.io/docs/"),
    ("ruptures change point detection",     "https://centre-borelli.github.io/ruptures-docs/"),
    ("Anthropic Tool Use",   "https://docs.anthropic.com/en/docs/build-with-claude/tool-use"),
    ("sklearn Anomaly Det.", "https://scikit-learn.org/stable/modules/outlier_detection.html"),
]
for name, url in resources:
    print(f"  • {name}: {url}")
print()
print("Lab 08 完成！恭喜完成 AIOps 實戰工作坊！")